# Práctica 2: Análisis Exploratorio de Datos (EDA)
## Actividad 6: Tratamiento de Desbalance - Dataset Titanic

En este notebook se evalúa el desbalance de la variable objetivo de supervivencia (`Survived`), se calcula el Imbalance Ratio (IR) y se comparan las 5 estrategias de tratamiento requeridas en la práctica.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# 1. Cargar el dataset del Titanic (asegúrate de que esté en tu carpeta 'data/')
df_titanic = pd.read_csv('../data/titanic.csv')

# Preprocesamiento básico de selección (convertir categóricas simples o eliminar nulos y texto no estructurado)
# Nota: Ajusta las columnas según el archivo CSV específico que estés utilizando
if 'Survived' in df_titanic.columns:
    target_col = 'Survived'
else:
    target_col = 'target' # En caso de que se llame distinto

# Seleccionamos variables numéricas relevantes para el ejemplo y limpiamos nulos
cols_num = df_titanic.select_dtypes(include=[np.number]).columns.tolist()
if target_col in cols_num:
    cols_num.remove(target_col)
if 'PassengerId' in cols_num:
    cols_num.remove('PassengerId')

df_titanic = df_titanic[cols_num + [target_col]].dropna()

X = df_titanic[cols_num]
y = df_titanic[target_col]

### 1. Análisis de Frecuencias y Cálculo del IR (Titanic)
Calculamos la cantidad de observaciones por clase, su porcentaje y la razón de desbalance ($IR$). Si $IR > 1.5$, se considera desbalanceado[cite: 3].

In [ ]:
conteo = y.value_counts()
porcentajes = y.value_counts(normalize=True) * 100

print("--- Conteo de observaciones ---")
print(conteo)
print("\n--- Porcentaje por clase ---")
print(porcentajes)

mayoritaria = conteo.idxmax()
minoritaria = conteo.idxmin()
IR = conteo[mayoritaria] / conteo[minoritaria]

print(f"\nRazón de desbalance (IR): {IR:.2f}")
if IR > 1.5:
    print("-> El dataset del Titanic presenta desbalance (IR > 1.5)[cite: 3].")
else:
    print("-> El dataset se encuentra razonablemente balanceado.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. Comparativa de Estrategias de Tratamiento
Se entrenan modelos de Regresión Logística evaluando: Línea base, Ponderación de clases, Sobremuestreo, Submuestreo y SMOTE[cite: 3].

In [ ]:
estrategias_titanic = {
    "1. Sin balanceo (Línea base)": (X_train, y_train, None),
    "2. Ponderación de clases": (X_train, y_train, "balanced"),
    "3. Sobremuestreo aleatorio": *RandomOverSampler(random_state=42).fit_resample(X_train, y_train), "none",
    "4. Submuestreo aleatorio": *RandomUnderSampler(random_state=42).fit_resample(X_train, y_train), "none",
    "5. SMOTE": *SMOTE(random_state=42).fit_resample(X_train, y_train), "none"
}

for nombre, datos in estrategias_titanic.items():
    if len(datos) == 3:
        X_tr, y_tr, cw = datos
        modelo = LogisticRegression(class_weight=cw, random_state=42, max_iter=1000)
    else:
        X_tr, y_tr, _ = datos
        modelo = LogisticRegression(random_state=42, max_iter=1000)
    
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_test)
    
    print(f"================ {nombre} ================")
    print(classification_report(y_test, y_pred, zero_division=0))